# Advanced Problems: Callables

Advanced exercises with solutions covering callable objects, functions, methods, classes, callable instances, stateful callables, decorators, and practical callable design patterns.

In [1]:
import inspect
import functools
from collections import defaultdict
from pprint import pprint


## Problem 1 — Classify Callable Objects

Write `classify_callable(obj)` that determines whether an object is callable and classifies it as a function, built-in function, method, class, callable instance, or non-callable.

In [2]:
def classify_callable(obj):
    if not callable(obj):
        return 'non-callable'
    if inspect.isfunction(obj):
        return 'function'
    if inspect.ismethod(obj):
        return 'bound method'
    if inspect.isbuiltin(obj):
        return 'built-in callable'
    if inspect.isclass(obj):
        return 'class'
    if hasattr(obj, '__call__'):
        return 'callable instance'
    return 'callable'


def f():
    pass

class A:
    def method(self):
        pass

    def __call__(self):
        return 'called'

items = [f, len, A, A().method, A(), 42, 'abc'.upper]

for item in items:
    print(repr(item), '=>', classify_callable(item))


<function f at 0x000002043C2E6660> => function
<built-in function len> => built-in callable
<class '__main__.A'> => class
<bound method A.method of <__main__.A object at 0x000002043C300AD0>> => bound method
<__main__.A object at 0x000002043C14D310> => callable instance
42 => non-callable
<built-in method upper of str object at 0x0000020436987C60> => built-in callable


## Problem 2 — Build a Stateful Callable Counter

Create a callable object `Counter` that increments internal state every time it is called.

In [3]:
class Counter:
    def __init__(self, start=0):
        self.value = start

    def __call__(self, step=1):
        self.value += step
        return self.value


counter = Counter(10)

print(callable(counter))
print(counter())
print(counter())
print(counter(5))
print(counter.value)


True
11
12
17
17


## Problem 3 — Compare Callable Classes and Callable Instances

Demonstrate that classes are callable because calling a class creates an instance, while instances are callable only if they implement `__call__`.

In [4]:
class Regular:
    pass


class CallableInstance:
    def __call__(self):
        return 'instance was called'


print('Regular class:', callable(Regular))
print('Regular instance:', callable(Regular()))

print('CallableInstance class:', callable(CallableInstance))
obj = CallableInstance()
print('CallableInstance instance:', callable(obj))
print(obj())


Regular class: True
Regular instance: False
CallableInstance class: True
CallableInstance instance: True
instance was called


## Problem 4 — Use Callable Objects as Configurable Functions

Build a callable `Power` object that behaves like a configured function.

In [5]:
class Power:
    def __init__(self, exponent):
        self.exponent = exponent

    def __call__(self, base):
        return base ** self.exponent


square = Power(2)
cube = Power(3)

print(square(5))
print(cube(5))
print(callable(square))
print(square.exponent)


25
125
True
2


## Problem 5 — Build a Callable Validator

Create a callable object that validates whether a value is within a configured numeric range.

In [6]:
class RangeValidator:
    def __init__(self, minimum=None, maximum=None):
        self.minimum = minimum
        self.maximum = maximum

    def __call__(self, value):
        if self.minimum is not None and value < self.minimum:
            return False
        if self.maximum is not None and value > self.maximum:
            return False
        return True


adult_age = RangeValidator(18, 120)

for age in [12, 18, 40, 150]:
    print(age, adult_age(age))


12 False
18 True
40 True
150 False


## Problem 6 — Create a Callable Registry

Build a registry that maps operation names to callables, then dispatches calls dynamically.

In [7]:
class CallableRegistry:
    def __init__(self):
        self._registry = {}

    def register(self, name, fn):
        if not callable(fn):
            raise TypeError('Only callables can be registered')
        self._registry[name] = fn

    def dispatch(self, name, *args, **kwargs):
        if name not in self._registry:
            raise KeyError(f'Unknown operation: {name}')
        return self._registry[name](*args, **kwargs)


registry = CallableRegistry()
registry.register('add', lambda a, b: a + b)
registry.register('upper', str.upper)

print(registry.dispatch('add', 10, 20))
print(registry.dispatch('upper', 'hello'))


30
HELLO


## Problem 7 — Make a Callable Decorator Class

Create a class-based decorator `CallLogger` that logs how many times a function has been called.

In [8]:
class CallLogger:
    def __init__(self, fn):
        functools.update_wrapper(self, fn)
        self.fn = fn
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        print(f'Calling {self.fn.__name__}; call #{self.calls}')
        return self.fn(*args, **kwargs)


@CallLogger
def multiply(a, b):
    return a * b


print(multiply(2, 3))
print(multiply(4, 5))
print(multiply.calls)
print(multiply.__name__)
print(callable(multiply))


Calling multiply; call #1
6
Calling multiply; call #2
20
2
multiply
True


## Problem 8 — Validate Function Arguments Before Calling

Write `safe_call(fn, *args, **kwargs)` that uses `inspect.signature` to validate arguments before executing the callable.

In [9]:
def safe_call(fn, *args, **kwargs):
    if not callable(fn):
        return {'ok': False, 'error': 'Object is not callable'}

    try:
        sig = inspect.signature(fn)
        bound = sig.bind(*args, **kwargs)
    except TypeError as ex:
        return {'ok': False, 'error': str(ex)}
    except ValueError:
        bound = None

    try:
        result = fn(*args, **kwargs)
    except Exception as ex:
        return {'ok': False, 'error': repr(ex)}

    return {
        'ok': True,
        'result': result,
        'bound_arguments': None if bound is None else dict(bound.arguments)
    }


def divide(a, b):
    return a / b

pprint(safe_call(divide, 10, 2))
pprint(safe_call(divide, 10))
pprint(safe_call(divide, 10, 0))
pprint(safe_call(42))


{'bound_arguments': {'a': 10, 'b': 2}, 'ok': True, 'result': 5.0}
{'error': "missing a required argument: 'b'", 'ok': False}
{'error': "ZeroDivisionError('division by zero')", 'ok': False}
{'error': 'Object is not callable', 'ok': False}


## Problem 9 — Implement a Retry Callable Wrapper

Create a callable wrapper that retries a function several times before raising the last exception.

In [10]:
class Retry:
    def __init__(self, fn, attempts=3):
        functools.update_wrapper(self, fn)
        self.fn = fn
        self.attempts = attempts

    def __call__(self, *args, **kwargs):
        last_error = None
        for attempt in range(1, self.attempts + 1):
            try:
                return self.fn(*args, **kwargs)
            except Exception as ex:
                print(f'Attempt {attempt} failed: {ex}')
                last_error = ex
        raise last_error


attempts = {'count': 0}

@Retry
def unstable():
    attempts['count'] += 1
    if attempts['count'] < 3:
        raise ValueError('temporary failure')
    return 'success'


print(unstable())
print(unstable.__name__)


Attempt 1 failed: temporary failure
Attempt 2 failed: temporary failure
success
unstable


## Problem 10 — Build a Callable Pipeline

Create a `Pipeline` class whose instances are callable. Each call should pass data through a sequence of callables.

In [11]:
class Pipeline:
    def __init__(self, *steps):
        for step in steps:
            if not callable(step):
                raise TypeError(f'{step!r} is not callable')
        self.steps = steps

    def __call__(self, value):
        for step in self.steps:
            value = step(value)
        return value


pipeline = Pipeline(str.strip, str.lower, lambda s: s.replace(' ', '-'))

print(pipeline('   Hello World From Python   '))
print(callable(pipeline))


hello-world-from-python
True


## Problem 11 — Count Calls Per Argument

Create a callable object that counts how often each argument value has been passed to it.

In [12]:
class FrequencyCounter:
    def __init__(self):
        self.counts = defaultdict(int)

    def __call__(self, value):
        self.counts[value] += 1
        return self.counts[value]


freq = FrequencyCounter()

for item in ['a', 'b', 'a', 'c', 'a', 'b']:
    print(item, '=>', freq(item))

print(dict(freq.counts))


a => 1
b => 1
a => 2
c => 1
a => 3
b => 2
{'a': 3, 'b': 2, 'c': 1}


## Problem 12 — Build a Mini Command System

Create a command system where commands are callable objects with names and descriptions.

In [13]:
class Command:
    def __init__(self, name, description, action):
        if not callable(action):
            raise TypeError('action must be callable')
        self.name = name
        self.description = description
        self.action = action

    def __call__(self, *args, **kwargs):
        return self.action(*args, **kwargs)


class CommandSystem:
    def __init__(self):
        self.commands = {}

    def register(self, command):
        if not callable(command):
            raise TypeError('command must be callable')
        self.commands[command.name] = command

    def run(self, name, *args, **kwargs):
        return self.commands[name](*args, **kwargs)

    def help(self):
        return {name: cmd.description for name, cmd in self.commands.items()}


system = CommandSystem()
system.register(Command('greet', 'Greet a user', lambda name: f'Hello, {name}!'))
system.register(Command('square', 'Square a number', lambda x: x * x))

print(system.run('greet', 'Simeon'))
print(system.run('square', 9))
pprint(system.help())


Hello, Simeon!
81
{'greet': 'Greet a user', 'square': 'Square a number'}


## Best-Practice Takeaways

- Use `callable(obj)` to test whether an object supports call syntax.
- Functions, methods, classes, and some instances are callable.
- Classes are callable because calling a class constructs an instance.
- Instances are callable only when their class defines `__call__`.
- Callable instances are useful when behavior needs persistent state.
- Class-based decorators should use `functools.update_wrapper` or `functools.wraps` to preserve metadata.
- Callable registries and pipelines are practical ways to build flexible, extensible systems.
- Always validate that user-supplied hooks, callbacks, or commands are callable before storing or executing them.